# Experiment 8: Building a POS Tagger

**Mapped Course Outcome:** CO3  
**BT Level:** BT2, BT3

## Aim
Build and evaluate a backoff part-of-speech (POS) tagger using default, unigram, bigram, and trigram tagging models.

## Objectives
- Prepare tagged corpus data for training and testing.
- Build statistical taggers of increasing context order.
- Use backoff to handle unseen contexts.
- Compare tagging accuracy and errors.

## Theory
A POS tagger assigns a grammatical category to every token in a sequence. A unigram tagger chooses the most likely tag for a word, while bigram and trigram taggers also use preceding predicted tags as context. Backoff combines these models: a higher-order tagger is tried first and falls back to a simpler tagger when it cannot decide. A default tagger supplies the final fallback for unknown words.
Evaluation uses tagged sentences that were not used during training.

## Methodology and Dataset
1. Load Brown corpus sentences with the Universal tagset.
2. Use a reproducible 90:10 train/test split.
3. Create a `DefaultTagger` fallback.
4. Train unigram, bigram, and trigram taggers with backoff.
5. Evaluate every model on the same held-out test set.
6. Tag a sample sentence and inspect mistakes.
The NLTK Brown corpus provides sentence boundaries and gold POS labels. A bounded subset is used so that the notebook remains convenient to run in a classroom or Colab environment.

In [1]:
import nltk
from nltk.corpus import brown
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger

nltk.download("brown", quiet=False)
nltk.download("universal_tagset", quiet=False)
print("NLTK resources are ready.")

[nltk_data] Downloading package brown to /home/omji/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/omji/nltk_data...


NLTK resources are ready.


[nltk_data]   Unzipping taggers/universal_tagset.zip.


In [2]:
# Load a deterministic subset and split it without mixing sentences.
MAX_SENTENCES = 12_000
all_sentences = brown.tagged_sents(tagset="universal")[:MAX_SENTENCES]
split_index = int(0.90 * len(all_sentences))
train_sents = all_sentences[:split_index]
test_sents = all_sentences[split_index:]

print(f"Total sentences: {len(all_sentences):,}")
print(f"Training sentences: {len(train_sents):,}")
print(f"Test sentences: {len(test_sents):,}")
print("Example:", train_sents[0])

Total sentences: 12,000
Training sentences: 10,800
Test sentences: 1,200
Example: [('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


In [3]:
# Build taggers from simple to complex. Each higher-order model backs off.
default_tagger = DefaultTagger("NOUN")
unigram_tagger = UnigramTagger(train_sents, backoff=default_tagger)
bigram_tagger = BigramTagger(train_sents, backoff=unigram_tagger)
trigram_tagger = TrigramTagger(train_sents, backoff=bigram_tagger)

taggers = {
    "Default": default_tagger,
    "Unigram": unigram_tagger,
    "Bigram + backoff": bigram_tagger,
    "Trigram + backoff": trigram_tagger,
}
print("Taggers trained:", ", ".join(taggers))

Taggers trained: Default, Unigram, Bigram + backoff, Trigram + backoff


In [4]:
import pandas as pd

results = []
for name, tagger in taggers.items():
    results.append({"Tagger": name, "Accuracy": tagger.accuracy(test_sents)})

comparison = pd.DataFrame(results)
comparison["Accuracy (%)"] = (comparison["Accuracy"] * 100).round(2)
comparison

,Tagger,Accuracy,Accuracy (%)
0,Default,0.238882,23.89
1,Unigram,0.919629,91.96
2,Bigram + backoff,0.928112,92.81
3,Trigram + backoff,0.927107,92.71


## Sample Sentence Tagging
The trigram tagger is used below. Unknown words and unseen contexts are handled by its unigram, then default, backoff chain.

In [5]:
sample_sentence = "The quick brown fox jumps over the lazy dog .".split()
sample_tags = trigram_tagger.tag(sample_sentence)
for word, tag in sample_tags:
    print(f"{word:>10} -> {tag}")

       The -> DET
     quick -> ADJ
     brown -> NOUN
       fox -> NOUN
     jumps -> NOUN
      over -> ADP
       the -> DET
      lazy -> ADJ
       dog -> NOUN
         . -> .


In [6]:
# Inspect the first 20 errors made by the trigram-backoff tagger.
errors = []
for sentence in test_sents:
    words = [word for word, _ in sentence]
    gold_tags = [tag for _, tag in sentence]
    predicted_tags = [tag for _, tag in trigram_tagger.tag(words)]
    for word, gold, predicted in zip(words, gold_tags, predicted_tags):
        if gold != predicted:
            errors.append({"Word": word, "Gold": gold, "Predicted": predicted})
    if len(errors) >= 20:
        break

pd.DataFrame(errors[:20])

,Word,Gold,Predicted
0,reborn,VERB,NOUN
1,incorruptible,ADJ,NOUN
2,Amen,PRT,NOUN
3,amen,PRT,NOUN
4,to,ADP,PRT
5,As,ADV,ADP
6,practice,VERB,NOUN
7,sin,NOUN,VERB
8,practice,VERB,NOUN
9,sin,NOUN,VERB


## Results and Discussion
The comparison table reports held-out accuracy for all models. Higher-order models can use more context, but their contexts are sparser. Backoff improves coverage by allowing unseen trigrams to use a bigram, unseen bigrams to use a unigram, and unknown words to use the default tag. Exact accuracy depends on the selected corpus subset and split.
## Outcomes
- Construct an n-gram POS-tagging pipeline.
- Explain default, unigram, bigram, trigram, and backoff tagging.
- Evaluate a tagger using held-out data.
- Analyze the trade-off between context and sparsity.
## Resources
- [NLTK Book, Chapter 5](https://www.nltk.org/book/ch05.html)
- [NLTK tag API](https://www.nltk.org/api/nltk.tag.html)
- [Jurafsky and Martin, Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)
- [Demonstrative video search](https://www.youtube.com/results?search_query=NPTEL+build+POS+tagger+unigram+bigram+backoff+NLP)
## Self-Assessment Questions
1. Define POS tagging.
2. What is a unigram tagger?
3. How does a bigram tagger use context?
4. What is a trigram tagger?
5. Explain backoff tagging.
6. Why is a default tagger useful?
7. What is a tagged corpus?
8. Why must training and test data be separate?
9. What causes unknown-word errors?
10. How would a neural POS tagger differ from this experiment?